# Тема 5. Задача 1. Обновление справочника тегов

## О чём вообще задача (простыми словами)

В мессенджере пользователи помечают свои сообщения **тегами** (например `#музыка`, `#спорт`). Есть список **проверенных, общедоступных тегов** — это `tags_verified`. Их уже одобрили.

Наша цель — найти **новых кандидатов в справочник**: теги, которые:

1. **ещё не входят** в проверенные (`tags_verified`);
2. набрали популярность — их предложило **много разных пользователей** (порог, например 100);
3. считаем за определённый **период** (7 или 84 дня).

Если тег популярный и его ещё нет в справочнике — он кандидат на добавление.

### Откуда берём данные

| Что | Путь | Что внутри |
|---|---|---|
| Сообщения | `/user/<user>/data/events/date=ГГГГ-ММ-ДД/event_type=message` | сообщения, а в них теги |
| Проверенные теги | `/user/master/data/snapshots/tags_verified/actual` | «белый список» тегов |

Данные **партиционированы по дате** — то есть для каждого дня своя папка `date=...`. Это удобно: чтобы прочитать 7 дней, мы берём ровно 7 папок, а не фильтруем гигантский датасет.

> ⚠️ Прод (`/user/prod`) ещё не готов, поэтому вместо него используем личную папку `/user/s24268544`.

---
# Задание 1. Функция `input_paths` — собираем пути за нужную глубину

**Задача:** написать функцию, которая по дате и глубине (числу дней) вернёт **список путей** к нужным партициям.

**Зачем:** чтобы не читать весь датасет и потом фильтровать, мы заранее формируем точные пути нужных дней. Это намного быстрее.

**Идея:** берём входную дату и идём назад по дням: день 0 (сама дата), день −1, день −2, ... до `depth` дней.

In [ ]:
from datetime import datetime, timedelta


def input_paths(date, depth):
    # 1) превращаем строку '2022-05-31' в объект даты, с которым можно делать арифметику
    dt = datetime.strptime(date, '%Y-%m-%d')

    # 2) для каждого смещения offset = 0, 1, ..., depth-1
    #    вычитаем offset дней и собираем путь к партиции этого дня
    return [
        f"/user/s24268544/data/events/"
        f"date={(dt - timedelta(days=offset)).strftime('%Y-%m-%d')}/"
        f"event_type=message"
        for offset in range(depth)
    ]

### Построчно

- `datetime.strptime(date, '%Y-%m-%d')` — парсит строку даты в дату. `%Y-%m-%d` — это формат `год-месяц-день`.
- `timedelta(days=offset)` — «сдвиг» на `offset` дней. `dt - timedelta(days=2)` = позавчера.
- `.strftime('%Y-%m-%d')` — обратно превращает дату в строку нужного формата для имени папки.
- `range(depth)` — даёт числа `0, 1, ..., depth-1`, то есть ровно `depth` дней.
- Списковое включение `[... for offset in range(depth)]` собирает список из `depth` путей.

Проверим на примере (можно запустить прямо тут):

In [ ]:
for p in input_paths('2022-05-31', 7):
    print(p)

Должно получиться 7 путей: `date=2022-05-31`, `date=2022-05-30`, ... до `date=2022-05-25`.

---
# Задание 2. Метрика за 7 дней в PySpark

**Задача:** посчитать теги-кандидаты за **7 дней** и записать в Parquet.

**Результат:** датафрейм с двумя колонками:
- `tag` — сам тег;
- `suggested_count` — сколько **уникальных** пользователей его предложили.

**Условия отбора:**
1. тег **не** из `tags_verified`;
2. период — 7 дней;
3. тег предложили **≥ 100** уникальных пользователей.

### Как устроены сообщения

Каждая запись событий имеет вложенную структуру `event`, внутри которой:
- `event.tags` — массив тегов (`array<string>`);
- `event.message_from` — кто отправил (это и есть «пользователь, предложивший тег»);
- `event.message_channel_to` — заполнено, если сообщение отправлено **в канал** (нам нужны именно такие).

In [ ]:
from datetime import datetime, timedelta

import pyspark.sql.functions as F
from pyspark.sql import SparkSession


def input_paths(date, depth):
    dt = datetime.strptime(date, '%Y-%m-%d')
    return [
        f"/user/s24268544/data/events/"
        f"date={(dt - timedelta(days=offset)).strftime('%Y-%m-%d')}/"
        f"event_type=message"
        for offset in range(depth)
    ]


# создаём точку входа в Spark; master='yarn' = считаем на кластере
spark = SparkSession.builder \
    .master("yarn") \
    .appName("candidates_d7_pyspark") \
    .getOrCreate()

# читаем сразу 7 партиций. Звёздочка *paths разворачивает список в отдельные аргументы
paths = input_paths('2022-05-31', 7)
messages = spark.read.parquet(*paths)

# читаем белый список проверенных тегов
verified_tags = spark.read.parquet("/user/master/data/snapshots/tags_verified/actual")

candidates = messages \
    .where("event.message_channel_to is not null") \
    .selectExpr("event.message_from as user", "explode(event.tags) as tag") \
    .groupBy("tag") \
    .agg(F.countDistinct("user").alias("suggested_count")) \
    .where("suggested_count >= 100") \
    .join(verified_tags, "tag", "left_anti")

candidates.write \
    .mode("overwrite") \
    .parquet("/user/s24268544/data/analytics/candidates_d7_pyspark")

### Разбор цепочки преобразований (самое важное!)

Читаем сверху вниз — каждый шаг получает результат предыдущего:

1. **`.where("event.message_channel_to is not null")`**
   Оставляем только сообщения, отправленные в канал (по подсказке задания — у них это поле непустое).

2. **`.selectExpr("event.message_from as user", "explode(event.tags) as tag")`**
   - `explode(event.tags)` — разворачивает массив тегов: если в сообщении 3 тега, получится 3 строки.
   - Было: `(пользователь, [тег1, тег2])` → стало: `(пользователь, тег1)` и `(пользователь, тег2)`.
   - `as user` / `as tag` — даём колонкам понятные имена.

3. **`.groupBy("tag")`** — группируем по тегу, чтобы посчитать по каждому отдельно.

4. **`.agg(F.countDistinct("user").alias("suggested_count"))`**
   Считаем **уникальных** пользователей на тег. `countDistinct` важен: если один человек поставил тег 10 раз — это всё равно 1 пользователь.

5. **`.where("suggested_count >= 100")`** — оставляем только теги, набравшие 100+ уникальных пользователей.

6. **`.join(verified_tags, "tag", "left_anti")`**
   `left_anti` — это «антиджойн»: оставляет из левой таблицы только те строки, для которых **нет** совпадения в правой. То есть **выкидывает** теги, которые уже есть в `tags_verified`. Остаются только новые.

7. **`.write.mode("overwrite").parquet(...)`** — сохраняем результат. `overwrite` = перезаписать, если уже есть.

---
# Задание 3. То же самое, но за 84 дня

Тут меняется буквально **две вещи** относительно Задания 2:
- глубина `7` → `84`;
- путь записи `candidates_d7_pyspark` → `candidates_d84_pyspark`.

Логика метрики не меняется. Смысл задания — посмотреть, как алгоритм поведёт себя на большем объёме данных (84 партиции читаются и считаются дольше).

In [ ]:
paths = input_paths('2022-05-31', 84)          # <-- 84 вместо 7
messages = spark.read.parquet(*paths)

verified_tags = spark.read.parquet("/user/master/data/snapshots/tags_verified/actual")

candidates = messages \
    .where("event.message_channel_to is not null") \
    .selectExpr("event.message_from as user", "explode(event.tags) as tag") \
    .groupBy("tag") \
    .agg(F.countDistinct("user").alias("suggested_count")) \
    .where("suggested_count >= 100") \
    .join(verified_tags, "tag", "left_anti")

candidates.write \
    .mode("overwrite") \
    .parquet("/user/s24268544/data/analytics/candidates_d84_pyspark")  # <-- d84

### Проверка результатов через `describe()`

`df.describe().show()` выводит статистику по колонкам: `count` (сколько строк), `mean`, `stddev`, `min`, `max`.

Для ответов нас интересует колонка `suggested_count`:
- `count` — сколько всего тегов-кандидатов отобрали;
- `min` — минимум уникальных пользователей;
- `max` — максимум.

Полученные результаты:

| Глубина | count (тегов) | min | max |
|---|---|---|---|
| 7 дней | 6 | 105 | 434 |
| 84 дня | 210 | 100 | 5440 |

Видно: за 84 дня кандидатов сильно больше (210 против 6) — данных больше, теги успевают набрать аудиторию.

In [ ]:
df = spark.read.parquet("/user/s24268544/data/analytics/candidates_d7_pyspark")
df.describe().show()

---
# Задание 4. Параметризованная джоба `verified_tags_candidates.py`

**Задача:** превратить «жёстко прошитый» код в **универсальный модуль**, которому всё передаётся параметрами: дата, глубина, порог, пути входа/выхода. Тогда одну и ту же джобу можно гонять на любые дату/глубину/порог.

**Имя приложения** должно быть `VerifiedTagsCandidatesJob-[дата]-d[глубина]-cut[порог]`, например `VerifiedTagsCandidatesJob-2022-05-31-d5-cut300`.

**Структура модуля (8 пунктов плана):**
1. импорты;
2. функция `main`;
3. чтение параметров из командной строки (`sys.argv`);
4. создание Spark-сессии и имя джобы;
5. чтение входных данных;
6. расчёт выходного DataFrame;
7. запись;
8. вызов `main()`.

In [ ]:
# ===== verified_tags_candidates.py =====
import sys
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
import pyspark.sql.functions as F


def input_paths(date, depth, events_base_path):
    # теперь базовый путь к событиям тоже параметр — чтобы легко переключать личную папку на prod
    dt = datetime.strptime(date, '%Y-%m-%d')
    return [
        f"{events_base_path}/"
        f"date={(dt - timedelta(days=offset)).strftime('%Y-%m-%d')}/"
        f"event_type=message"
        for offset in range(depth)
    ]


def main():
    # ----- 3) параметры из командной строки -----
    # sys.argv[0] — имя самого скрипта, поэтому аргументы начинаются с [1]
    date = sys.argv[1]                  # '2022-05-31'
    depth = int(sys.argv[2])            # 5   (строку приводим к числу)
    threshold = int(sys.argv[3])        # 300
    events_base_path = sys.argv[4]      # путь к событиям
    verified_tags_path = sys.argv[5]    # путь к проверенным тегам
    output_path = sys.argv[6]           # куда писать результат

    # ----- 4) Spark-сессия с динамическим именем -----
    spark = SparkSession.builder \
        .master("yarn") \
        .appName(f"VerifiedTagsCandidatesJob-{date}-d{depth}-cut{threshold}") \
        .getOrCreate()

    # ----- 5) чтение входных данных -----
    messages = spark.read.parquet(*input_paths(date, depth, events_base_path))
    verified_tags = spark.read.parquet(verified_tags_path)

    # ----- 6) расчёт (та же метрика, что в Задании 2/3) -----
    candidates = messages \
        .where("event.message_channel_to is not null") \
        .selectExpr("event.message_from as user", "explode(event.tags) as tag") \
        .groupBy("tag") \
        .agg(F.countDistinct("user").alias("suggested_count")) \
        .where(f"suggested_count >= {threshold}") \
        .join(verified_tags, "tag", "left_anti") \
        .withColumn("date", F.lit(date))   # добавляем колонку date для партиционирования

    # ----- 7) запись с партиционированием по дате -----
    # partitionBy('date') создаёт подпапку output_path/date=2022-05-31
    candidates.write \
        .mode("overwrite") \
        .partitionBy("date") \
        .parquet(output_path)


# ----- 8) запуск -----
if __name__ == "__main__":
    main()

### Что нового по сравнению с Заданиями 2–3

- **`sys.argv`** — список аргументов командной строки. Когда мы запускаем `spark-submit verified_tags_candidates.py 2022-05-31 5 300 ...`, эти значения попадают в `argv[1]`, `argv[2]` и т.д. `int(...)` нужен, потому что из командной строки всё приходит строками.
- **`appName(f"...-{date}-d{depth}-cut{threshold}")`** — f-строка подставляет параметры в имя, чтобы в YARN было видно, какой именно запуск.
- **`.withColumn("date", F.lit(date))`** — добавляем колонку с датой. `F.lit(...)` = «литерал», одно и то же значение для всех строк.
- **`.partitionBy("date")`** — при записи Spark создаёт подпапку `date=2022-05-31`. Сама колонка `date` уходит в имя папки, в файлах остаются только `tag` и `suggested_count`.

### Как запускать (на кластере)

```bash
/usr/lib/spark/bin/spark-submit --master yarn --deploy-mode cluster \
  verified_tags_candidates.py 2022-05-31 5 300 \
  /user/s24268544/data/events \
  /user/master/data/snapshots/tags_verified/actual \
  /user/s24268544/data/analytics/verified_tags_candidates_d5
```

Аргументы по порядку: `дата  глубина  порог  путь_событий  путь_проверенных_тегов  путь_результата`.

> **Важно:** файл `verified_tags_candidates.py` должен физически лежать на ноде кластера (там, откуда запускаешь `spark-submit`), а не только на ноутбуке.

---
# Задание 5. Автоматизация в Airflow (DAG)

**Задача:** чтобы не запускать джобу руками каждый день, добавляем её в **DAG** (граф задач Airflow). Airflow будет запускать всё по расписанию.

**Что нужно:**
- взять DAG из прошлой темы (там была таска `partitioning` — перекладка сырых данных в Parquet);
- добавить **две** новые таски: за 7 и за 84 дня;
- сделать их **зависимыми** от `partitioning` — ведь наша джоба читает то, что перекладка только что записала.

**Зависимость** записывается так:
```python
partitioning >> [verified_tags_candidates_d7, verified_tags_candidates_d84]
```
Стрелка `>>` означает «сначала это, потом то». Список справа = обе таски запустятся **после** перекладки (и параллельно друг другу).

In [ ]:
# ===== dag.py =====
import os
import datetime as dt

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperator

# переменные окружения, нужные Spark/Hadoop
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['JAVA_HOME'] = '/usr'
os.environ['SPARK_HOME'] = '/usr/lib/spark'
os.environ['PYTHONPATH'] = '/usr/local/lib/python3.8'

default_args = {
    'owner': 's24268544',
    'start_date': dt.datetime(2022, 5, 1),
    'retries': 1,                              # 1 повтор при падении
    'retry_delay': dt.timedelta(minutes=5),    # пауза 5 минут перед повтором
}

with DAG(
    'partition_events',
    default_args=default_args,
    schedule_interval='@daily',   # запуск раз в день
    catchup=False,                # не догонять пропущенные прошлые даты
) as dag:

    # таска из прошлой темы — перекладка сырых событий в Parquet
    partitioning = BashOperator(
        task_id='partitioning',
        bash_command=(
            'spark-submit --master yarn --deploy-mode cluster '
            '/lessons/partition.py 2022-05-31 '
            '/user/master/data/events /user/s24268544/data/events'
        ),
    )

    # новая таска: кандидаты за 7 дней
    verified_tags_candidates_d7 = SparkSubmitOperator(
        task_id='verified_tags_candidates_d7',
        application='/lessons/verified_tags_candidates.py',  # путь к джобе на кластере
        conn_id='yarn_spark',                                # соединение Airflow со Spark/YARN
        application_args=[
            '2022-05-31', '7', '100',
            '/user/s24268544/data/events',
            '/user/master/data/snapshots/tags_verified/actual',
            '/user/s24268544/data/analytics/verified_tags_candidates_d7',
        ],
    )

    # новая таска: кандидаты за 84 дня
    verified_tags_candidates_d84 = SparkSubmitOperator(
        task_id='verified_tags_candidates_d84',
        application='/lessons/verified_tags_candidates.py',
        conn_id='yarn_spark',
        application_args=[
            '2022-05-31', '84', '100',
            '/user/s24268544/data/events',
            '/user/master/data/snapshots/tags_verified/actual',
            '/user/s24268544/data/analytics/verified_tags_candidates_d84',
        ],
    )

    # сначала перекладка, потом обе джобы кандидатов
    partitioning >> [verified_tags_candidates_d7, verified_tags_candidates_d84]

### Что важно понять про DAG

- **`DAG`** — это описание «что за чем запускать». Сам код задач не выполняется в Airflow — Airflow только дирижирует.
- **`BashOperator`** — запускает bash-команду (тут `spark-submit ...` строкой).
- **`SparkSubmitOperator`** — специальный оператор для Spark-джоб. Удобнее: путь к скрипту в `application`, аргументы списком в `application_args`, соединение в `conn_id`.
- **`conn_id='yarn_spark'`** — заранее настроенное в Airflow подключение к Spark/YARN. Если у тебя оно называется иначе — поправь.
- **`>>`** — задаёт зависимости. `A >> B` = «B после A». Так Airflow знает порядок.

### Схема нашего DAG

```
                       ┌─> verified_tags_candidates_d7
   partitioning  ──────┤
                       └─> verified_tags_candidates_d84
```

Перекладка готовит данные, затем обе джобы-кандидата читают их и считают метрику за свои глубины.

> При релизе в прод достаточно поменять `/user/s24268544/...` на `/user/prod/data/...` — структура аргументов та же.

---
# Итог: что мы построили

| Задание | Что сделали | Файл |
|---|---|---|
| 1 | Функция путей за нужную глубину | `realization.py` |
| 2 | Расчёт кандидатов за 7 дней | `realization.py` |
| 3 | Расчёт за 84 дня + проверка `describe()` | `realization.py` |
| 4 | Универсальная параметризованная джоба | `verified_tags_candidates.py` |
| 5 | Автоматизация запуска в Airflow | `dag.py` |

**Главная мысль всей задачи:** мы строим конвейер, который регулярно находит новые популярные теги (которых ещё нет в справочнике) и складывает их кандидатами для пополнения `tags_verified`. Сначала прототип на конкретной дате (Задания 1–3), потом — гибкий модуль (Задание 4), потом — автоматизация по расписанию (Задание 5).